### 1) Implementing attention mechanisms in an NLP model (TensorFlow/Keras)

In [23]:
import tensorflow as tf 
from tensorflow import keras 
from tensorflow.keras import layers 
import numpy as np 

class BahdanauAttention(layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.w1 = layers.Dense(units)
        self.w2 = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, query, keys, values):
        query = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query)+self.W2(keys)))
        weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(weights*values, axis=1)
        return context, weights 

# Encoder: Embedding. + GRU
def build_encoder(vocab_size, emb_dim, hidden_dim):
    inputs = keras.Input(shape=(None, ))
    x = layers.Embedding(vocab_size, emb_dim)(inputs)
    outputs, state = layers.GRU(hidden_dim, return_sequences= True)
    return keras.Model(inputs, [outputs, state], name='encoder')

# Decoder: Embedding + GRU + Attention + Dense 
def build_decoder(vocab_size, emb_dim, hidden_dim):
    dec_input = keras.Input(shape=(1, ))
    enc_outputs = keras.Input(shape=(None, hidden_dim))
    dec_state = keras.Input(shape=(hidden_dim))

    x = layers.Embedding(vocab_size, emb_dim)(dec_input)
    x = layers.GRU(hidden_dim, return_sequences=False, return_state=False)(x,initial_state=dec_state)



SyntaxError: incomplete input (309280243.py, line 28)

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# Positional encoding: PE(pos,2i) = sin(pos/10000^(2i/d)), PE(pos,2i+1) = cos(pos/10000^(2i/d))
class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()
        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(d_model)[np.newaxis, :]
        angles = pos / np.power(10000, (2 * (i // 2)) / d_model)
        self.pe = np.zeros((max_len, d_model))
        self.pe[:, 0::2] = np.sin(angles[:, 0::2])
        self.pe[:, 1::2] = np.cos(angles[:, 1::2])
        self.pe = tf.cast(self.pe[np.newaxis, :, :], tf.float32)
    
    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

# Transformer encoder block
class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout=0.1):
        super().__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn = keras.Sequential([
            layers.Dense(dff, activation='relu'),
            layers.Dense(d_model)
        ])
        self.ln1 = layers.LayerNormalization()
        self.ln2 = layers.LayerNormalization()
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)
    
    def call(self, x, training=False):
        attn_out = self.mha(x, x, x)
        attn_out = self.dropout1(attn_out, training=training)
        x = self.ln1(x + attn_out)
        ffn_out = self.ffn(x)
        ffn_out = self.dropout2(ffn_out, training=training)
        return self.ln2(x + ffn_out)

# Build classifier: Embedding -> PosEnc -> TransformerBlock -> GlobalAvgPool -> Dense
def build_transformer_classifier(vocab_size, max_len, d_model, num_heads, dff, num_classes):
    inputs = keras.Input(shape=(max_len,))
    x = layers.Embedding(vocab_size, d_model)(inputs)
    x = PositionalEncoding(max_len, d_model)(x)
    x = TransformerBlock(d_model, num_heads, dff)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(num_classes, activation='softmax' if num_classes > 2 else 'sigmoid')(x)
    return keras.Model(inputs, outputs)

# Synthetic dataset: classify sequences by their sum modulo 3
vocab_size, max_len, num_classes = 20, 10, 3
X_train = np.random.randint(1, vocab_size, (300, max_len))
y_train = np.sum(X_train, axis=1) % num_classes
X_val = np.random.randint(1, vocab_size, (50, max_len))
y_val = np.sum(X_val, axis=1) % num_classes

# Build and compile model
model = build_transformer_classifier(
    vocab_size=vocab_size,
    max_len=max_len,
    d_model=32,
    num_heads=4,
    dff=64,
    num_classes=num_classes
)
model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Train for 1 epoch
model.fit(X_train, y_train, batch_size=32, epochs=1, validation_data=(X_val, y_val))

# Evaluate
test_loss, test_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"\nValidation accuracy: {test_acc:.3f}")

# Example prediction
sample = X_val[0:1]
pred = model.predict(sample, verbose=0)
print(f"Sample input: {sample[0][:5]}... (sum%3={y_val[0]})")
print(f"Predicted class: {np.argmax(pred[0])}, probabilities: {pred[0]}")

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 10, 32)         │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ (None, 10, 32)         │             0 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 10, 32)         │         8,544 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,283 (36.26 KB)

 Trainable params: 9,283 (36.26 KB)

 Non-trainable params: 0 (0.00 B)

10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.3200 - loss: 1.3555 - val_accuracy: 0.2400 - val_loss: 1.2532

Validation accuracy: 0.240
Sample input: [13  2  4 16  7]... (sum%3=1)
Predicted class: 0, probabilities: [0.51117027 0.15895468 0.32987508]


In [16]:
# pip install transformers torch

import torch
import torch.nn as nn
from transformers import DistilBertModel, AutoTokenizer
import numpy as np

# Load pre-trained DistilBERT and tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert = DistilBertModel.from_pretrained(model_name)

# Freeze BERT weights for faster training (optional)
for param in bert.parameters():
    param.requires_grad = False

# Build classifier: BERT -> pooled output -> Dropout -> Dense
class BertClassifier(nn.Module):
    def __init__(self, bert_model, num_classes=2):
        super().__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(bert_model.config.hidden_size, num_classes)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]  # CLS token
        x = self.dropout(pooled)
        return self.classifier(x)

# Tiny dataset: sentiment classification (positive/negative)
texts = [
    "This movie is absolutely wonderful and entertaining!",
    "Terrible film, waste of time and money.",
    "I loved every minute of it, highly recommended.",
    "Boring and poorly made, very disappointed."
]
labels = torch.tensor([1, 0, 1, 0])  # 1=positive, 0=negative

# Tokenize texts
encoded = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors='pt'
)

# Build model
model = BertClassifier(bert, num_classes=2)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("\nTraining...")

# Train for 1 epoch
model.train()
for epoch in range(1):
    optimizer.zero_grad()
    outputs = model(encoded['input_ids'], encoded['attention_mask'])
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    acc = (outputs.argmax(dim=1) == labels).float().mean()
    print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Accuracy={acc.item():.3f}")

# Single prediction
model.eval()
test_text = ["This is an amazing experience!"]
test_encoded = tokenizer(test_text, padding=True, truncation=True, max_length=32, return_tensors='pt')

with torch.no_grad():
    logits = model(test_encoded['input_ids'], test_encoded['attention_mask'])
    probs = torch.softmax(logits, dim=1)
    prediction = logits.argmax(dim=1)

print(f"\nTest: '{test_text[0]}'")
print(f"Prediction: {'Positive' if prediction[0] == 1 else 'Negative'}")
print(f"Probabilities: Negative={probs[0][0]:.3f}, Positive={probs[0][1]:.3f}")

Model parameters: 66,364,418

Training...
Epoch 1: Loss=0.7015, Accuracy=0.500

Test: 'This is an amazing experience!'
Prediction: Positive
Probabilities: Negative=0.498, Positive=0.502


In [22]:
!pip install scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 3.8 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 2.4 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [19]:
# pip install transformers torch pandas scikit-learn

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import pandas as pd
from sklearn.model_selection import train_test_split

# Simulated CSV-like dataset: product reviews with ratings
data = {
    'text': [
        "Excellent product, highly recommended!",
        "Terrible quality, broke after one day.",
        "Good value for money, works as expected.",
        "Waste of money, very disappointed.",
        "Amazing! Exceeded my expectations.",
        "Poor design and cheap materials.",
        "Perfect for my needs, very satisfied.",
        "Not worth the price, returned it.",
        "Best purchase I've made this year!",
        "Awful experience, would not recommend.",
        "Great quality and fast shipping.",
        "Defective item, poor customer service.",
    ],
    'label': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]  # 1=positive, 0=negative
}
df = pd.DataFrame(data)

# Split dataset
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.25, random_state=42
)

# Load tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Custom Dataset class
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length)
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Create datasets and dataloaders
train_dataset = ReviewDataset(train_texts, train_labels, tokenizer)
val_dataset = ReviewDataset(val_texts, val_labels, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

# Build classifier with pre-trained BERT
class BertClassifier(nn.Module):
    def __init__(self, model_name, num_classes=2, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]  # CLS token
        x = self.dropout(pooled)
        return self.classifier(x)

# Initialize model, optimizer, loss
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BertClassifier(model_name, num_classes=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

print(f"Device: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}\n")

# Training loop (1 epoch)
model.train()
total_loss, correct, total = 0, 0, 0

for batch in train_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)
    
    optimizer.zero_grad()
    outputs = model(input_ids, attention_mask)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    total_loss += loss.item()
    predictions = outputs.argmax(dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

print(f"Training - Loss: {total_loss/len(train_loader):.4f}, Accuracy: {correct/total:.3f}")

# Evaluation
model.eval()
val_correct, val_total = 0, 0

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask)
        predictions = outputs.argmax(dim=1)
        val_correct += (predictions == labels).sum().item()
        val_total += labels.size(0)

print(f"Validation Accuracy: {val_correct/val_total:.3f}")

# Single prediction example
test_text = "This product is fantastic and works perfectly!"
test_encoding = tokenizer(test_text, return_tensors='pt', truncation=True, max_length=64)
test_encoding = {k: v.to(device) for k, v in test_encoding.items()}

with torch.no_grad():
    logits = model(**test_encoding)
    probs = torch.softmax(logits, dim=1)
    prediction = logits.argmax(dim=1)

print(f"\nTest: '{test_text}'")
print(f"Predicted: {'Positive' if prediction[0] == 1 else 'Negative'}")
print(f"Confidence: {probs[0][prediction[0]].item():.3f}")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
import tensorflow as tf
from tensorflow import keras 
from tensorflow.keras import layers 
import numpy as np 

class PositionEncoding(layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()
        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(d_model)[np.newaxis, :]
        angles = pos/np.power(10000, (2*(i//2))/d_model)
        self.pe = np.zeros((max_len, d_model))
        self.pe[:, 0::2] = np.sin(angles[:, 0::2])
        self.pe[:, 1::2] = np.cos(angles[:, 1::2])
        self.pe = tf.cast(self.pe[np.newaxis, : , :], tf.float32)

    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]
    
# transformer encoder block
class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout=0.1):
        super().__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model//num_heads)
        self.ffn = keras.Sequential([
            layers.Dense(dff, activation='relu'),
            layers.Dense(d_model)
        ])
        self.ln1 = layers.LayerNormalization()
        self.ln2 = layers.LayerNormalization()
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn_out = self.mha(x, x, x)
        attn_out = self.dropout1(attn_out, training=training)
        x = self.ln1(x+attn_out)
        ffn_out = self.ffn(x)
        ffn_out = self.dropout2(ffn_out, training=training)
        return self.ln2(x+ffn_out) 
    
def build_transformer_classifier(vocab_size, max_len, d_model, num_heads, dff, num_classes):
    inputs = keras.Input(shape=(max_len, ))
    x = layers.Embedding(vocab_size, d_model)(inputs)
    x = PositionalEncoding(max_len, d_model)(x)
    x = TransformerBlock(d_model, num_heads, dff)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(num_classes, activation='softmax' if num_classes > 2 else 'sigmoid')(x)
    return keras.Model(inputs, outputs)


vocab_size, max_len, num_classes = 20, 10, 3
x_train = np.random.randint(1,vocab_size, (300, max_len))
y_train = np.sum(X_train, axis=1) % num_classes 
X_val = np.random.randint(1, vocab_size, (50, max_len))
y_val = np.sum(X_val, axis=1) % num_classes

model = build_transformer_classifier(vocab_size=vocab_size, max_len=max_len, d_model=32, num_heads=4, dff=64, num_classes=num_classes)

model.compile(optimizer=keras.optimizers.Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [37]:
# pip install transformers torch

import torch
import torch.nn as nn
from transformers import DistilBertModel, AutoTokenizer
import numpy as np

# Load pre-trained DistilBERT and tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert = DistilBertModel.from_pretrained(model_name)

# Freeze BERT weights for faster training (optional)
for param in bert.parameters():
    param.requires_grad = False

# Build classifier: BERT -> pooled output -> Dropout -> Dense
class BertClassifier(nn.Module):
    def __init__(self, bert_model, num_classes=2):
        super().__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(bert_model.config.hidden_size, num_classes)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]  # CLS token
        x = self.dropout(pooled)
        return self.classifier(x)

# Tiny dataset: sentiment classification (positive/negative)
texts = [
    "This movie is absolutely wonderful and entertaining!",
    "Terrible film, waste of time and money.",
    "I loved every minute of it, highly recommended.",
    "Boring and poorly made, very disappointed."
]
labels = torch.tensor([1, 0, 1, 0])  # 1=positive, 0=negative

# Tokenize texts
encoded = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors='pt'
)

# Build model
model = BertClassifier(bert, num_classes=2)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("\nTraining...")

# Train for 1 epoch
model.train()
for epoch in range(1):
    optimizer.zero_grad()
    outputs = model(encoded['input_ids'], encoded['attention_mask'])
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    acc = (outputs.argmax(dim=1) == labels).float().mean()
    print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Accuracy={acc.item():.3f}")

# Single prediction
model.eval()
test_text = ["This is an amazing experience!"]
test_encoded = tokenizer(test_text, padding=True, truncation=True, max_length=32, return_tensors='pt')

with torch.no_grad():
    logits = model(test_encoded['input_ids'], test_encoded['attention_mask'])
    probs = torch.softmax(logits, dim=1)
    prediction = logits.argmax(dim=1)

print(f"\nTest: '{test_text[0]}'")
print(f"Prediction: {'Positive' if prediction[0] == 1 else 'Negative'}")
print(f"Probabilities: Negative={probs[0][0]:.3f}, Positive={probs[0][1]:.3f}")

Model parameters: 66,364,418

Training...
Epoch 1: Loss=0.7413, Accuracy=0.250

Test: 'This is an amazing experience!'
Prediction: Negative
Probabilities: Negative=0.597, Positive=0.403


In [47]:
import torch 
import torch.nn as nn

from transformers import DistilBertModel, AutoTokenizer 
import numpy as np 

model_name = 'distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_name )

bert = DistilBertModel.from_pretrained(model_name)

for param in bert.parameters(): 
    param.requires_grad = False 

class BertClassifier(nn.Module):
    
    def __init__(self, bert_model, num_classes=2):
        super().__init__()
        self.bert = bert_model 
        self.dropout=nn.Dropout(0.1)

        self.classifier = nn.Linear(bert_model.config.hidden_size, num_classes)

    def forward(self, input_ids, atten_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=atten_mask)

        pooled = outputs.last_hidden_state[:, 0, :]

        x = self.dropout(pooled)

        return self.classifier(x)
    
texts = [
    "This movie is absolutely wonderful and entertaining!",
    "Terrible film, waste of time and money.",
    "I loved every minute of it, highly recommended.",
    "Boring and poorly made, very disappointed."
]


In [48]:
labels = torch.tensor([1, 0, 1, 0], dtype=torch.long)

encoded = tokenizer(texts, padding=True, truncation=True, max_length=32, return_tensors='pt')

model = BertClassifier(bert, num_classes =2)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

criterion = nn.CrossEntropyLoss()

model.train()

for epoch in range(1):
    optimizer.zero_grad()

    outputs = model(encoded['input_ids'], encoded['attention_mask'])

    loss = criterion(outputs, labels)

    loss.backward()

    optimizer.step()

    acc = (outputs.argmax(dim=1) == labels).float().mean()
    print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Accuracy={acc.item():.3f}")

model.eval()

Epoch 1: Loss=0.7842, Accuracy=0.000


BertClassifier(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(

In [ ]:
test_text = 